In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [4]:
load_dotenv() 
subgraph_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=1024)

In [5]:
class SubState(TypedDict):

    input_text: str
    translated_text: str

In [8]:
def translate_text(state: SubState):

    prompt = f"""
Translate the following text into Bangla. 
Ensure the translation is natural and clear without adding extra information.

Text:
{state["input_text"]}
""".strip()
    
    translated_text = subgraph_model.invoke(prompt).content

    return {'translated_text': translated_text}

In [11]:
subgraph = StateGraph(SubState)

subgraph.add_node('translate_text', translate_text)

subgraph.add_edge(START, 'translate_text')
subgraph.add_edge('translate_text', END)

subgraph = subgraph.compile()

In [12]:
class ParentState(TypedDict):

    question: str
    answer_eng: str
    answer_ban: str
    

In [13]:
parent_model = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0, max_tokens=1024)

In [14]:
def generate_answer(state: ParentState):

    answer = parent_model.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}